# DBSCAN — Example



## 1 · Dataset
We generate a **synthetic dataset** consisting of a dense central cluster, two concentric rings, and randomly distributed noise points.

This dataset illustrates DBSCAN's ability to identify **non-convex clusters** and detect noise. Since DBSCAN relies on distance calculations, we standardize the features before applying the algorithm.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN, KMeans

SEED = 42
rng = np.random.default_rng(SEED)

# Create two noisy orbital rings, with evenly spaced angles to avoid large gaps.
ring_points = []
for radius, n_points, thickness in [(1.45, 170, 0.07), (3.15, 230, 0.09)]:
    angles = np.linspace(0, 2 * np.pi, n_points, endpoint=False)
    angles += rng.normal(0, 0.009, n_points)
    ring = np.column_stack([radius * np.cos(angles), radius * np.sin(angles)])
    ring += rng.normal(0, thickness, size=ring.shape)
    ring_points.append(ring)

# Add a dense central group and randomly distributed background objects.
center = rng.normal(0, 0.24, size=(100, 2))
background = rng.uniform(-4.2, 4.2, size=(25, 2))
X_raw = np.vstack([*ring_points, center, background])
X = StandardScaler().fit_transform(X_raw)

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], s=16, color="#64748b", alpha=0.75)
plt.title("Unlabeled observations: orbital rings and background")
plt.xlabel("Feature 1 (standardized)")
plt.ylabel("Feature 2 (standardized)")
plt.axis("equal")
plt.show()

## 2 · Run DBSCAN
- `eps` defines the radius of each neighborhood.
- `min_samples` counts the minimum observations for a **core point**, including the point itself.
- Label `-1` indicates **noise**.

Notice that we never tell DBSCAN how many clusters to find.

In [ ]:
eps = 0.24          # Try 0.12, 0.24, 0.40
min_samples = 5     # Try 5 and 10

# Discover density-connected clusters; -1 means noise.
labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X)
n_clusters = len(set(labels) - {-1})
n_noise = np.sum(labels == -1)
print(f"Clusters: {n_clusters} | Noise points: {n_noise}")

plt.figure(figsize=(6, 5))
colors = plt.get_cmap("tab10")
for label in sorted(set(labels)):
    mask = labels == label
    color = "#9ca3af" if label == -1 else colors(label % 10)
    name = "Noise" if label == -1 else f"Cluster {label + 1}"
    plt.scatter(X[mask, 0], X[mask, 1], s=18, color=color, label=name, alpha=0.85)
plt.title(f"DBSCAN: eps={eps}, min_samples={min_samples}")
plt.xlabel("Feature 1 (standardized)")
plt.ylabel("Feature 2 (standardized)")
plt.axis("equal")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

## 3 · Compare with K-Means
Both algorithms see exactly the same unlabeled observations. For a fair illustration, K-Means is given **three clusters**; DBSCAN is not given a cluster count.

In [ ]:
kmeans_labels = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit_predict(X)
dbscan_labels = DBSCAN(eps=0.24, min_samples=5).fit_predict(X)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharex=True, sharey=True)
for ax, lab, title in zip(axes, [kmeans_labels, dbscan_labels], ["K-Means (K=3)", "DBSCAN"]):
    for cluster in sorted(set(lab)):
        mask = lab == cluster
        color = "#9ca3af" if cluster == -1 else plt.get_cmap("tab10")(cluster % 10)
        ax.scatter(X[mask, 0], X[mask, 1], s=15, color=color, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel("Feature 1")
    ax.set_aspect("equal")
axes[0].set_ylabel("Feature 2")
plt.tight_layout()
plt.show()

## Exercices
1. Try `eps = 0.12` and then `eps = 0.40`. Do rings split into fragments or merge with the center?
2. Restore `eps = 0.24` and change `min_samples` from **5** to **10**. How does the noise count change?
3. Why does K-Means split the rings into angular sectors instead of identifying the rings themselves?

**Takeaway:** DBSCAN can discover non-convex, density-connected structures and label isolated observations as noise. Parameter choices still matter.